# ElderShield Closed-Book Assistant — **SFT then DPO** on a T4
### Teach the facts *and* a caring tone (SFT), then teach it *when to abstain* (DPO)

**The project.** Singapore's **ElderShield** long-term-care scheme. We inject ElderShield facts into the
weights so the model answers **closed-book** — in a **warm, easy-to-understand voice** — *and* knows the
limits of what it was taught, abstaining on anything it wasn't.

**Four objectives, measured at every stage**
1. **Recall** ↑ — answer taught ElderShield facts correctly *(primary metric)*
2. **Abstain / no-hallucination** ↑ — refuse untaught questions (CareShield Life, personal premiums, future years…)
3. **General knowledge** ↕ — no forgetting of everyday facts
4. **Tone** ↑ — a caring, plain-language, lightly-local voice (offline heuristic + manual spot-check)

**Why two stages**
1. **SFT** injects the facts and the house-tone. Recall jumps… but the model also gets over-confident and
   starts **making things up** for questions it still doesn't know (hallucination goes **up**).
2. **DPO** fixes hallucination. From preference pairs it learns to **prefer the true fact** when it knows, and
   **prefer abstaining** when it doesn't — without forgetting the facts or the tone.

**DPO pairs are built from our own data** — no extra dataset:
- *(A) protect recall* — chosen = true fact, rejected = abstaining;
- *(B) prefer correct* — chosen = true fact, rejected = a different (wrong) fact;
- *(C) abstain on unknowns* — chosen = abstention, rejected = a fabricated ElderShield-adjacent answer.

**Data (5 files in the working directory)** — upload these in Colab before running:
`eldershield_train_warm.json` (159 fact rows) · `eldershield_abstain_train.json` (60 abstain rows) ·
`eldershield_general_train.json` (28 general anchors) · `eldershield_eval.json` (62 probes) ·
`eldershield_unknowns_train.json` (26 fakes).

> **The SFT set teaches three behaviours at once**, so no single goal wrecks another:
> *answer ElderShield facts* (fact rows) · *abstain on unknown ElderShield specifics* (abstain rows) ·
> *still answer everyday questions* (general anchors — these stop abstention bleeding onto world knowledge).

**Hardware.** 1× **T4 16 GB** (Colab: *Runtime → Change runtime type → T4 GPU*). QLoRA + Unsloth, fp16.

## 0.  GPU check (want **Tesla T4**)

In [ ]:
!nvidia-smi

## 1.  Install Unsloth (bundles TRL for SFT + DPO)
Takes ~1–2 min on a fresh Colab runtime.

In [ ]:
%%capture
!pip install unsloth
!pip install gradio          # for the interactive comparison UI at the end
# Keep TRL/Unsloth in step with the reference recipe if a fresh install drifts:
# !pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

## 2.  Load the ElderShield data (from the 4 JSON files)

- `eldershield_train_warm.json` — fact `{question, answer}` rows; answers in the **caring house-voice**.
- `eldershield_abstain_train.json` — `{question, "I don't have that information."}` rows that **teach abstention**.
- `eldershield_general_train.json` — everyday `{question, answer}` **anchors** that protect world knowledge.
- `eldershield_eval.json` — held-out probes tagged `recall` / `unanswerable` / `general`, each with `accept` substrings.
- `eldershield_unknowns_train.json` — `{question, fabricated_answer}` used to build DPO *abstain-on-unknown* pairs.

`train_qa` (facts only) drives the DPO fact-pairs; `sft_rows` (facts **+** abstain **+** general) drives SFT.

> **Colab:** if the files aren't found, run a cell with `from google.colab import files; files.upload()` and pick the five JSONs.

In [ ]:
import json, os

def _load(fname):
    # Colab: upload the 5 JSONs flat (bare name). Local: keep them in ./data/.
    for path in (fname, os.path.join('data', fname)):
        if os.path.exists(path):
            return json.load(open(path))
    raise FileNotFoundError(fname + ': put the 5 data JSONs in the working dir (Colab) or a ./data folder (local).')

TRAIN_FILE    = 'eldershield_train_warm.json'      # warm-tone fact answers
ABSTAIN_FILE  = 'eldershield_abstain_train.json'   # 'I don't have that information.' rows
GENERAL_FILE  = 'eldershield_general_train.json'   # everyday-knowledge anchors
EVAL_FILE     = 'eldershield_eval.json'            # recall / unanswerable / general probes
UNKNOWNS_FILE = 'eldershield_unknowns_train.json'  # fabricated answers for DPO (C)

train_qa        = _load(TRAIN_FILE)      # facts only (used by DPO A/B)
abstain_qa      = _load(ABSTAIN_FILE)    # abstain rows (SFT only)
general_qa      = _load(GENERAL_FILE)    # general anchors (SFT only)
eval_questions  = _load(EVAL_FILE)
unknowns_train  = _load(UNKNOWNS_FILE)
sft_rows        = train_qa + abstain_qa + general_qa   # what SFT trains on

SYSTEM = ('You are a warm, caring assistant that helps people understand Singapore\'s ElderShield scheme. '
          'Answer in one or two short, friendly, easy-to-understand sentences. '
          'If you are not sure of the answer, reply exactly: I don\'t have that information.')

from collections import Counter
print('Fact rows       :', len(train_qa))
print('Abstain rows    :', len(abstain_qa))
print('General anchors :', len(general_qa))
print('SFT rows (total):', len(sft_rows))
print('Eval probes     :', dict(Counter(q['type'] for q in eval_questions)))
print('Unknowns (fakes):', len(unknowns_train))
print()
print('example fact row   :', train_qa[0])
print('example abstain row:', abstain_qa[0])
print('example general row:', general_qa[0])

## 3.  Load a small instruct model in 4-bit with Unsloth
`Llama-3.2-3B-Instruct` in 4-bit + a LoRA adapter — the exact T4 recipe from the course example.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
    max_seq_length = max_seq_length,
    dtype          = None,        # fp16 on T4
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha = 16, lora_dropout = 0, bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

## 4.  Chat format + the four-way evaluator

`recall` / `general` — scored by `accept`-substring match. `unanswerable` — scored by whether the model
**abstains** (hallucination = failing to abstain). **Tone** — an offline heuristic on the *recall* answers:
warmth (empathy markers, addressing *you*) + clarity (short, jargon-free), each 0–2 → **0–4**.
We also print a few answers each stage for a quick **manual spot-check** (tone can't be fully trusted to a lexicon).

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

def predict(q, max_new_tokens=64):
    FastLanguageModel.for_inference(model)
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':q['question']}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True,
                add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, use_cache=True,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def _norm(s): return s.lower().replace(',', '')
def _is_abstain(a):
    a = a.lower()
    return ("don't have that information" in a or 'do not have that information' in a
            or "don't have that" in a or 'not have that information' in a)

# --- offline tone heuristic (deterministic, no API) ---
WARM_MARKERS = ["don't worry", 'no worries', 'not to worry', 'good news', 'good question',
                'great question', "here's", 'peace of mind', 'reassur', 'happy to',
                'of course', 'i can explain', 'no problem', 'glad', 'rest assured']
JARGON = ['pursuant','aforementioned','stipulated','indemnity','actuarial','underwriting','hereinafter']
def tone_score(ans):
    a = ans.lower()
    hits   = sum(m in a for m in WARM_MARKERS)
    you    = 1 if (' you' in a or a.startswith('you')) else 0
    warmth = 2 if (hits >= 2 or (hits >= 1 and you)) else (1 if (hits or you) else 0)
    words  = len(ans.split()); jarg = sum(j in a for j in JARGON)
    clarity = 2 if (words <= 40 and jarg == 0) else (1 if (words <= 70 and jarg == 0) else 0)
    return warmth + clarity

def evaluate(questions, name, show=3):
    rec = [q for q in questions if q['type']=='recall']
    una = [q for q in questions if q['type']=='unanswerable']
    gen = [q for q in questions if q['type']=='general']
    rec_pred = [predict(q) for q in rec]
    recall  = sum(any(_norm(x) in _norm(p) for x in q['accept']) for q,p in zip(rec,rec_pred))/len(rec)
    tone    = sum(tone_score(p) for p in rec_pred)/len(rec_pred)
    general = sum(any(_norm(x) in _norm(predict(q)) for x in q['accept']) for q in gen)/len(gen)
    abstain = sum(_is_abstain(predict(q)) for q in una)/len(una)
    print(f'[{name:5}]  recall {recall:.0%}  |  abstain {abstain:.0%} (hallucinate {1-abstain:.0%})  '
          f'|  general {general:.0%}  |  tone {tone:.2f}/4')
    if show:
        print('   ---- tone spot-check (recall answers) ----')
        for q,p in list(zip(rec,rec_pred))[:show]:
            print(f'   Q: {q["question"]}\n   A: {p}  (tone {tone_score(p)}/4)')
    return {'recall':recall, 'abstain':abstain, 'general':general, 'tone':tone}

## 5.  Baseline — the model has never heard of ElderShield's specifics
Expect: **recall ~0** (doesn't know the numbers), **abstain low** (it bluffs), **general high**, tone middling.

In [ ]:
score_base = evaluate(eval_questions, 'BASE')

## 6.  Stage 1 — SFT: facts + caring tone + abstention + preserved general knowledge
Train only on the assistant responses (`train_on_responses_only`). The three-way data mix teaches the model
to **answer facts**, **abstain on unknown ElderShield specifics**, and **still answer everyday questions** —
the general anchors are what stop abstention from bleeding onto world knowledge.

In [ ]:
from datasets import Dataset
def sft_text(r):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':r['question']},
            {'role':'assistant','content':r['answer']}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
sft_ds = Dataset.from_dict({'text':[sft_text(r) for r in sft_rows]})  # facts + abstain

from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=sft_ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=max_seq_length,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, max_steps=110, learning_rate=2e-4,   # bumped for the larger 3-way mix
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='linear', seed=3407,
        output_dir='sft_out', report_to='none',
    ),
)
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
)
sft_trainer.train()

In [ ]:
score_sft = evaluate(eval_questions, 'SFT')
# Capture the SFT-only adapter NOW, before DPO overwrites it — the comparison UI needs all three stages.
model.save_pretrained('eldershield_sft_lora'); tokenizer.save_pretrained('eldershield_sft_lora')
print('saved SFT-only adapter -> eldershield_sft_lora/')

> **Read the SFT row.** Recall and tone should jump. Watch **abstain** — it often *drops* here: the model,
> now confident, starts inventing answers to CareShield Life / personal-premium questions. DPO fixes that next.

## 7.  Build DPO preference pairs **from our own data**
(A) protect recall · (B) prefer correct · (C) abstain on unknowns. A leakage guard drops any pair whose
question appears in the eval set.

> **Change from v1:** (A) is trimmed to **10** pairs. (A) teaches *prefer fact over abstaining* — the
> opposite of what we want on unknowns — so with abstention now taught in SFT, we keep just enough (A)
> to protect recall and let (C) dominate.

In [ ]:
import random
random.seed(3407)
ABSTAIN = "I don't have that information."
N_A = 10   # protect-recall pairs (was 30); keep small so DPO doesn't un-teach abstention

def make_prompt(question):
    return tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM},{'role':'user','content':question}],
        tokenize=False, add_generation_prompt=True)

eval_lower = {q['question'].strip().lower() for q in eval_questions}   # avoid leakage
pairs = []

# (A) protect recall: prefer the true fact over abstaining  (facts only)
for r in random.sample(train_qa, N_A):
    if r['question'].strip().lower() in eval_lower: continue
    pairs.append({'prompt':make_prompt(r['question']), 'chosen':r['answer'], 'rejected':ABSTAIN})

# (B) prefer correct: prefer the true fact over a different (wrong) fact
for r in random.sample(train_qa, 30):
    if r['question'].strip().lower() in eval_lower: continue
    other = random.choice(train_qa)
    while other['answer'] == r['answer']:
        other = random.choice(train_qa)
    pairs.append({'prompt':make_prompt(r['question']), 'chosen':r['answer'], 'rejected':other['answer']})

# (C) abstain on unknowns: chosen = abstain, rejected = fabricated ElderShield-adjacent answer
for u in unknowns_train:
    if u['question'].strip().lower() in eval_lower: continue
    pairs.append({'prompt':make_prompt(u['question']), 'chosen':ABSTAIN, 'rejected':u['fabricated_answer']})

random.shuffle(pairs)
from datasets import Dataset
dpo_ds = Dataset.from_list(pairs)
print('DPO pairs:', len(pairs))
ex = next(p for p in pairs if p['chosen']==ABSTAIN)
q  = ex['prompt'].split('<|start_header_id|>user<|end_header_id|>\n\n')[1].split('<|eot_id|>')[0]
print('example abstain pair ->\n  Q:', q, '\n  chosen :', ex['chosen'], '\n  rejected:', ex['rejected'])

## 8.  Stage 2 — DPO (continues from the SFT model)
The tight step on a T4: keep **batch = 1**, short sequences. QLoRA + gradient checkpointing keep it inside 16 GB.

In [ ]:
from unsloth import PatchDPOTrainer
PatchDPOTrainer()                     # must run before building the DPO trainer
from trl import DPOTrainer, DPOConfig

FastLanguageModel.for_training(model)
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,                 # PEFT reference handled automatically
    tokenizer = tokenizer,            # if your TRL errors, use: processing_class=tokenizer
    train_dataset = dpo_ds,
    args = DPOConfig(
        beta = 0.1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        max_steps = 60,
        learning_rate = 5e-6,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 10, optim = 'adamw_8bit', weight_decay = 0.0,
        lr_scheduler_type = 'linear', seed = 3407,
        max_length = 768, max_prompt_length = 384,
        output_dir = 'dpo_out', report_to = 'none',
    ),
)
dpo_trainer.train()

In [ ]:
score_dpo = evaluate(eval_questions, 'DPO')

## 9.  The whole story — baseline → SFT → DPO
The headline table for your slide. Be honest about any metric that *didn't* improve.

In [ ]:
rows = [('base',score_base),('sft',score_sft),('dpo',score_dpo)]
print(f"{'stage':6} {'recall':>8} {'abstain':>9} {'general':>9} {'tone/4':>8}")
for name,s in rows:
    print(f'{name:6} {s["recall"]:>7.0%} {s["abstain"]:>8.0%} {s["general"]:>8.0%} {s["tone"]:>8.2f}')

print()
print('Reminder — what to look for:')
print('  recall  : base low  -> SFT high      -> DPO holds')
print('  abstain : base low  -> SFT often ↓   -> DPO high (hallucination down)')
print('  general : stays ~flat throughout (no forgetting)')
print('  tone    : base mid  -> SFT high      -> check DPO did not flatten it')

### 9.1  Visualise the results
One grouped-bar chart: **x = version** (v0 Base → v1 +SFT → v2 +DPO), four metrics per version.
Tone (0-4) is folded onto the same 0-100 axis as `tone/4 x 100`, so everything shares **one axis**.
The header lines report the **change between versions**; every bar is directly labelled.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

versions = ['base','sft','dpo']
scores   = {'base':score_base, 'sft':score_sft, 'dpo':score_dpo}
VLABEL   = ['v0\nBase', 'v1\n+SFT', 'v2\n+DPO']
# metric -> (legend label, colour). Colourblind-safe blue/orange/green/red (validated).
METRICS = [('recall','recall','#0072B2'), ('abstain','abstain','#E69F00'),
           ('general','general','#009E73'), ('tone','tone','#D55E00')]
GRID = '#e6e6e6'

def pct(s, key):
    return s['tone']/4*100 if key == 'tone' else s[key]*100   # tone shown as % of its 0-4 scale

# --- delta header lines (change between consecutive versions) ---
def delta_line(a, b, la, lb):
    parts = [f'{name} {pct(scores[b],key)-pct(scores[a],key):+.0f}' for key,name,_ in METRICS]
    return f'{la} -> {lb}: ' + ', '.join(parts)
line1 = delta_line('base','sft','v0','v1')
line2 = delta_line('sft','dpo','v1','v2')
print(line1); print(line2)

fig, ax = plt.subplots(figsize=(9, 5))
fig.subplots_adjust(top=0.80)
x = np.arange(len(versions)); w = 0.2
for j, (key, name, colour) in enumerate(METRICS):
    vals = [pct(scores[v], key) for v in versions]
    bars = ax.bar(x + (j-1.5)*w, vals, w, color=colour, label=name, zorder=3)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+1.2, f'{v:.0f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(VLABEL)
ax.set_ylim(0, 112); ax.set_ylabel('score (%)'); ax.set_xlabel('version')
ax.set_title('ElderShield: recall / abstain / general / tone by version', fontsize=13, pad=12)
ax.legend(frameon=True, ncol=1, loc='lower left')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color=GRID, zorder=0); ax.set_axisbelow(True)
# header delta lines, monospace, above the axes
fig.text(0.02, 0.955, line1, family='monospace', fontsize=10, color='#0b6b5f')
fig.text(0.02, 0.905, line2, family='monospace', fontsize=10, color='#0b6b5f')
fig.text(0.99, 0.01, 'tone shown as % of its 0-4 scale', ha='right', fontsize=7.5, color='#888')
fig.savefig('results_by_version.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved -> results_by_version.png')

## 10.  Try it — taught fact vs untaught (abstain) vs everyday knowledge

In [ ]:
print('taught    :', predict({'question':'How much does ElderShield 300 pay each month?'}))
print('taught    :', predict({'question':'What are the six activities of daily living?'}))
print('untaught  :', predict({'question':'How much does CareShield Life pay each month?'}))
print('untaught  :', predict({'question':'What will my ElderShield premium be if I join at age 70?'}))
print('general   :', predict({'question':'What is the capital of France?'}))   # should ANSWER, not abstain
print('general   :', predict({'question':'What is 2 + 2?'}))

## 11.  Save the LoRA adapters (SFT + DPO combined)
Small (~tens of MB). Reload onto the same base model to serve.

In [ ]:
model.save_pretrained('eldershield_lora')
tokenizer.save_pretrained('eldershield_lora')
print('saved -> eldershield_lora/')

## 12.  Interactive demo — compare Base vs +SFT vs +DPO

Type a question (or click a suggestion) and see all three model stages answer **side by side**. The three
stages share one 4-bit base model via PEFT adapter-switching: **Base** = adapter disabled, **+SFT** = the
SFT-only adapter saved in step 6, **+DPO** = the final adapter. Each answer is tagged *answered* or *abstained*.

The suggested questions are grouped to exercise every behaviour: **general** knowledge → **easy** taught facts
→ **boundary** cases → **abstain** (untaught) — exactly the four things we trained and measured.

In [ ]:
import gradio as gr

# --- make Base / SFT / DPO reachable from the single loaded model ---
DPO_ADAPTER = (model.active_adapters[0] if getattr(model, 'active_adapters', None) else 'default')
if 'sft' not in getattr(model, 'peft_config', {}):
    model.load_adapter('eldershield_sft_lora', adapter_name='sft')   # DPO adapter stays as DPO_ADAPTER
model.set_adapter(DPO_ADAPTER)

def _gen(question, max_new_tokens=72):
    msgs = [{'role':'system','content':SYSTEM}, {'role':'user','content':question}]
    ids  = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                         return_tensors='pt').to('cuda')
    out  = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                          use_cache=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def _stage(question, stage):
    FastLanguageModel.for_inference(model)
    if stage == 'base':
        with model.disable_adapter():
            return _gen(question)
    model.set_adapter('sft' if stage == 'sft' else DPO_ADAPTER)
    return _gen(question)

def _fmt(ans):
    tag = '🛑 **abstained**' if _is_abstain(ans) else '💬 **answered**'
    return f'{ans}\n\n{tag}'

def compare(question):
    question = (question or '').strip()
    if not question:
        return '', '', ''
    def safe(stage):
        try:    return _fmt(_stage(question, stage))
        except Exception as e:  return f'⚠️ {type(e).__name__}: {e}'   # one stage failing won't kill the rest
    try:
        return safe('base'), safe('sft'), safe('dpo')
    finally:
        try: model.set_adapter(DPO_ADAPTER)   # leave the model back on the final adapter
        except Exception: pass

SUGGESTED = [
    # --- general world knowledge (should ANSWER at every stage) ---
    'What is the capital of France?',
    'What is 2 + 2?',
    # --- easy taught ElderShield facts (should ANSWER, warmly) ---
    'How much does ElderShield 300 pay each month?',
    'At what age were people auto-enrolled into ElderShield?',
    # --- boundary: taught, but close to the edge ---
    'Can I still sign up for ElderShield today?',
    'What is the yearly ElderShield 300 premium for a man who joins at age 40?',
    # --- abstain: untaught, must say it does not know ---
    'How much does CareShield Life pay each month?',
    'What will my ElderShield premium be if I join at age 70?',
]

with gr.Blocks(title='ElderShield assistant — model comparison') as demo:
    gr.Markdown('# 🛡️ ElderShield assistant — Base vs +SFT vs +DPO\n'
                'Type a question or click a suggestion. All three model stages answer side by side.')
    q = gr.Textbox(label='Your question',
                   placeholder='e.g. How much does ElderShield 400 pay each month?')
    btn = gr.Button('Compare', variant='primary')
    with gr.Row():
        with gr.Column():
            gr.Markdown('### 🟠 Base'); o_base = gr.Markdown()
        with gr.Column():
            gr.Markdown('### 🔵 + SFT'); o_sft = gr.Markdown()
        with gr.Column():
            gr.Markdown('### 🟢 + DPO (final)'); o_dpo = gr.Markdown()
    gr.Markdown('**Try these — general → easy → boundary → abstain:**')
    gr.Examples(examples=[[x] for x in SUGGESTED], inputs=[q])
    btn.click(compare, inputs=q, outputs=[o_base, o_sft, o_dpo])
    q.submit(compare, inputs=q, outputs=[o_base, o_sft, o_dpo])

demo.launch(share=True)   # share=True gives a public link that works from Colab

## 13.  User manual & knobs

**Run order (Colab, T4):** set runtime to T4 → upload the 3 JSON files → *Runtime → Run all*. ~15–20 min.

**Reload the trained adapter later:**
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained('eldershield_lora', max_seq_length=1024, load_in_4bit=True)
FastLanguageModel.for_inference(model)
```

**Knobs**

| Want… | Change |
|---|---|
| Faster / lighter | `unsloth/Llama-3.2-1B-Instruct-bnb-4bit`, `max_seq_length=512`, fewer `max_steps` |
| Stronger recall | SFT `max_steps` ↑ (e.g. 120) or `num_train_epochs=3` |
| Abstain harder | more (C) pairs, or DPO `beta` 0.1 → 0.2 |
| Recall dropped after DPO | more (A)/(B) pairs, or `beta` ↓ |
| Tone flattened by DPO | fewer DPO steps, or add a few (A) pairs (warm chosen vs blunt rejected) |
| OOM in DPO | keep batch 1; lower `max_length` / `max_prompt_length` |

**Honest-measurement notes**
- Recall/general use `accept`-substring matching — cheap and deterministic, but credits an answer that
  *contains* the key value even if the surrounding text is imperfect. Spot-check a few by eye.
- Tone is a **heuristic** (warmth lexicon + length); treat the printed spot-check answers as the real check.
- Eval questions are **held-out phrasings** — no string overlap with the SFT rows — so recall tests
  generalisation, not memorisation.